In [58]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [60]:
df = pd.read_csv("/Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/DataChecks/prefect_logs/prefect_logs_2026-02-18 07:39:50.816955.csv")

# Analyze Date Ranage

In [61]:
range1 = df[(df["timestamp"]>="2026-01-19")&(df["timestamp"]<"2026-01-26")]
range1["date"] = pd.to_datetime(range1["timestamp"]).dt.date
range2 = df[(df["timestamp"]>="2026-02-09")&(df["timestamp"]<"2026-02-16")]
range2["date"] = pd.to_datetime(range2["timestamp"]).dt.date


In [62]:
range1["name"].value_counts().sort_index()

In [63]:
range1.groupby("date")["name"].value_counts().sort_index()

In [64]:
range2["name"].value_counts().sort_index()

In [65]:
range2.groupby("date")["name"].value_counts().sort_index()

# Get Node sizes

In [66]:
path_to_save_data = "/Users/karolinegriesbach/Documents/Innkeepr/Analysen/StackITCosts/202502/"

In [67]:
range_data = pd.concat([range1, range2])
range_data["week"] = pd.to_datetime(range_data["timestamp"]).dt.isocalendar().week
print(range_data["node_name"].value_counts(dropna=False))
range_data["node_name"] = np.where((range_data["node_name"].isnull())&(range_data["type"]=="etl"), "etl_node", range_data["node_name"])
print(range_data["node_name"].value_counts(dropna=False))
range_data

In [68]:
# count of node sizes
count_node_usage = range_data.groupby("week")["node_name"].value_counts(dropna=True).reset_index()
print(count_node_usage)
count_node_usage_pivot = count_node_usage.pivot(index="node_name", columns="week", values="count").reset_index()
count_node_usage_pivot.to_csv(f"{path_to_save_data}count_node_usage.csv", index=False)
count_node_usage

In [69]:
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(1,1,1)
sns.barplot(data=count_node_usage, x="node_name", y="count", hue="week")
plt.title("Node count per week")
plt.grid(True)
plt.xticks(rotation=90)
plt.tight_layout()
fig.savefig(f"{path_to_save_data}node_count_per_week.png")

In [70]:
range_data["duration_in_min"] = range_data["duration"] / 60
duration_node_usage = range_data.groupby(by=["week","node_name"])["duration_in_min"].sum().reset_index()
duration_node_usage_pivot = duration_node_usage.pivot(index="node_name", columns="week", values="duration_in_min").reset_index()
duration_node_usage_pivot["diff (7-4)"] = duration_node_usage_pivot[7] - duration_node_usage_pivot[4]
duration_node_usage_pivot


In [71]:
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(1,1,1)
sns.barplot(data=duration_node_usage, x="node_name", y="duration_in_min", hue="week")
plt.title("Pod Runtime in Minutes per Week")
plt.grid(True)
plt.xticks(rotation=90)
plt.tight_layout()
fig.savefig(f"{path_to_save_data}duration_node_usage.png")

# Vergleich Node Ressourcen woche 4 (19.01 - 25.01.2026) und Woche 7 (09.02 - 15.02.2026)

- Count: Es wurden in Woche 4 nicht mehr große Instanzen verwendet als in Woche 7. Eher im Gegenteil, sogar weniger -> müsste eigentlich zur Kostenreduktion führen
- Runtime der Pods: 
    - wie wird es gemessen: end_time_pod - start_time_pod
    - hier kann die node länger laufen als der pod (das wird hier in den Metriken nicht mit abgebildet, dafür siehe Phil's logs)
    - Bis auf die 16GB die minimal mehr Laufzeit hat, reduziert sich hier die Laufzeit auch bei allen Pods -> spricht auch dafür, dass es günstiger werden sollte

Ich habe die Daten jetzt aus der Tabelle gezogen, aber bei [Mixpanel](https://eu.mixpanel.com/project/3086547/view/3601723/app/boards#id=7904460&edited-bookmark=uGYD2aMGThDR,) ist der Trend durchaus aus sichtbar, allerdings ist dort der etl Flow nicht mit abgebildet. Da werden zwar die Laufzeiten getrackt, aber nicht die node_names -> das müssen wir mal noch ändern, dass das auch abbildbar ist. 

## mit Fehlertoleranz
To simulate failed targetings

In [104]:
duration_node_usage_error_tolerance = duration_node_usage.copy()
error_tolerance = 0.24 # bei 24% mehr Laufzeit bei 32GB
duration_node_usage_error_tolerance["error_tolerance"] = error_tolerance
duration_node_usage_error_tolerance[f"duration_in_min + {error_tolerance*100}%"] = np.where(
    duration_node_usage_error_tolerance["week"] == 7,
    duration_node_usage_error_tolerance["duration_in_min"] * duration_node_usage_error_tolerance["error_tolerance"] + duration_node_usage_error_tolerance["duration_in_min"],
    duration_node_usage_error_tolerance["duration_in_min"]
)
duration_node_usage_error_tolerance

In [103]:
duration_node_usage_pivot_error_tolerance = duration_node_usage_error_tolerance.pivot(index="node_name", columns="week", values=f"duration_in_min + {error_tolerance*100}%").reset_index()
duration_node_usage_pivot_error_tolerance["diff (7-4)"] = duration_node_usage_pivot_error_tolerance[7] - duration_node_usage_pivot_error_tolerance[4]
duration_node_usage_pivot_error_tolerance

In [99]:
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(1,1,1)
sns.barplot(data=duration_node_usage_error_tolerance, x="node_name", y=f"duration_in_min + {error_tolerance*100}%", hue="week")
plt.title("Pod Runtime in Minutes per Week")
plt.grid(True)
plt.xticks(rotation=90)